**Study A baselines, one at a time**

Runs each no-atom baseline from `scripts/run_baselines.py` on its own, then
puts them side by side in one comparison table (the same layout as table 1 in
the notes) with paired significance tests.

Task: the facts section of a judgment is the question; the WGBO articles the
judgment cites are the answers. Every system scores all corpus articles for
each question; the ranking is evaluated with MRR, recall@k and nDCG@10.

Systems, in order:

1. **Random**: articles in random order, the floor.
2. **BM25**: keyword match against the article text.
3. **Dense (BGE-M3)**: embedding similarity between question and article.
4. **Hybrid, z-score fusion**: standardise both score lists per question, add them.
5. **Hybrid, reciprocal rank fusion (RRF)**: add up `1 / (60 + rank)` from each system.
6. **RRF + spreading** along citations, within legal sections, and both.

Runs on the **dev** split only. The test split is for the final run of
`run_baselines.py --split test --final`.

## Setup

In [1]:
import sys
import os
import json
from pathlib import Path
from types import SimpleNamespace


def find_project_root(marker='src/similarity/retrieval.py'):
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find project root (no {marker} in any parent)")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 220)

from src.graph.propagation import citation_adjacency, spread, tree_adjacency
from src.similarity.retrieval import (
    article_documents, bm25_scores, evaluate, random_scores, rrf_fusion, zscore_fusion,
)
from scripts.run_baselines import QUERIES, STATUTES, _dense

SPLIT = 'dev'
# Same settings as the run_baselines.py defaults (spreading strengths chosen on dev)
CONFIG = SimpleNamespace(model='BAAI/bge-m3', max_length=8192, random_draws=100,
                         beta_cite=0.25, beta_tree=1.0, beta_both=(0.1, 1.0))

In [2]:
if not QUERIES.exists():
    raise FileNotFoundError(f'{QUERIES} is missing: run scripts/build_study_a.py first')

items = [json.loads(line) for line in QUERIES.open(encoding='utf-8')]
items = [it for it in items if it['split'] == SPLIT]
questions = [it['question'] for it in items]
answers = [it['answers'] for it in items]
ids, docs = article_documents(STATUTES)
cite, tree = citation_adjacency(ids), tree_adjacency(ids, STATUTES)

print(f'{len(items)} {SPLIT} questions, {len(ids)} articles')
print(f'answers per question: mean {np.mean([len(a) for a in answers]):.2f}, '
      f'max {max(len(a) for a in answers)}')
print(f'citation links: {int(cite.sum() // 2)}, section links: {int(tree.sum() // 2)}')

125 dev questions, 87 articles
answers per question: mean 2.29, max 7
citation links: 48, section links: 644


In [3]:
scores = {}       # system -> questions x articles score matrix
per_query = {}    # system -> per-question metrics
summary = {}      # system -> mean metrics


def run(system, matrix):
    # Evaluate one system, keep its results and show its mean metrics
    pq, s = evaluate(matrix, ids, answers)
    scores[system], per_query[system], summary[system] = matrix, pq, s
    return pd.DataFrame([s], index=[system]).round(3)


def top_k(system, q=0, k=5):
    # The top k articles for question q, marking the correct ones
    order = np.argsort(-scores[system][q])[:k]
    return pd.DataFrame({'article': [ids[i] for i in order],
                         'score': scores[system][q, order].round(4),
                         'correct': [ids[i] in answers[q] for i in order]},
                        index=range(1, k + 1))

## 1. Random

Articles in random order: the floor every system must beat. With 87 articles
the expected recall@10 is about 10/87 ≈ 0.11. Averaged over 100 random orders.

In [4]:
runs = [evaluate(random_scores(len(items), len(ids), seed=s), ids, answers)
        for s in range(CONFIG.random_draws)]
per_query['random'] = sum(r[0] for r in runs) / len(runs)
summary['random'] = {k: round(float(np.mean([r[1][k] for r in runs])), 4) for k in runs[0][1]}
pd.DataFrame([summary['random']], index=['random']).round(3)

,mrr,recall@1,recall@5,recall@10,recall@20,ndcg@10
random,0.097,0.011,0.056,0.112,0.226,0.063


## 2. BM25

Keyword search: an article scores high when it shares rare words with the
facts. No training; the usual `k1 = 1.5, b = 0.75`.

In [5]:
run('bm25', bm25_scores(questions, docs))

,mrr,recall@1,recall@5,recall@10,recall@20,ndcg@10
bm25,0.325,0.081,0.271,0.463,0.608,0.293


In [6]:
top_k('bm25')

,article,score,correct
1,7:447,33.2357,False
2,6:175,16.5048,False
3,6:182,11.5570,False
4,7:450,10.1202,True
5,6:108,10.0252,False


## 3. Dense (BGE-M3)

Cosine similarity between embeddings of the question and of each article.
BGE-M3 reads up to 8,192 tokens, so most facts sections fit whole. Scores are
cached in `data/processed/study_a/cache/` (shared with `run_baselines.py`);
the first run embeds everything on CPU and takes a while.

In [7]:
run('dense', _dense(questions, docs, CONFIG))

dense: cached scores dense_BAAI_bge-m3_534d5e61d7fe.npy


,mrr,recall@1,recall@5,recall@10,recall@20,ndcg@10
dense,0.313,0.082,0.306,0.416,0.614,0.28


In [8]:
top_k('dense')

,article,score,correct
1,7:447,0.5520,False
2,6:183,0.5415,False
3,7:450,0.5156,True
4,7:465,0.5079,False
5,7:458a,0.4794,False


## 4. Hybrid: z-score fusion

BM25 scores and cosine similarities live on different scales. Standardising
each question's scores (mean 0, sd 1) makes them addable: an article ranks
high when both systems like it, or one system likes it a lot.

In [9]:
run('hybrid', zscore_fusion(scores['bm25'], scores['dense']))

,mrr,recall@1,recall@5,recall@10,recall@20,ndcg@10
hybrid,0.367,0.096,0.292,0.461,0.658,0.31


## 5. Hybrid: reciprocal rank fusion (RRF)

RRF throws away the scores and keeps only each system's **ranking**. Every
article gets

$$\text{RRF}(d) = \sum_{\text{systems}} \frac{1}{k + \text{rank}(d)}, \qquad k = 60$$

(Cormack et al., 2009). Because it only uses ranks, no scaling is needed and
there is nothing to tune. The constant `k` flattens the curve: rank 1 gives
1/61 ≈ 0.0164, rank 10 gives 1/70 ≈ 0.0143, so an article ranked well by
*both* systems beats one ranked first by one system and badly by the other.

A toy example with three articles:

In [10]:
toy_bm25 = np.array([[12.0, 3.0, 7.5]])      # BM25 ranks: a=1, b=3, c=2
toy_dense = np.array([[0.41, 0.52, 0.50]])   # dense ranks: a=3, b=1, c=2
pd.DataFrame({'bm25': toy_bm25[0], 'dense': toy_dense[0],
              'rrf': rrf_fusion(toy_bm25, toy_dense)[0].round(5)},
             index=['a', 'b', 'c'])

,bm25,dense,rrf
a,12.0,0.41,0.03227
b,3.0,0.52,0.03227
c,7.5,0.50,0.03226


Article *c* is second in both lists and wins: consistent beats extreme. On the real data:

In [11]:
run('hybrid_rrf', rrf_fusion(scores['bm25'], scores['dense']))

,mrr,recall@1,recall@5,recall@10,recall@20,ndcg@10
hybrid_rrf,0.376,0.101,0.3,0.463,0.658,0.318


In [12]:
top_k('hybrid_rrf')

,article,score,correct
1,7:447,0.0328,False
2,7:450,0.0315,True
3,6:183,0.0308,False
4,7:458a,0.0303,False
5,6:169,0.0303,False


## 6. RRF + spreading over links between articles

After RRF, each article passes part of its score (scaled to [0, 1] per
question) to linked articles: `s' = s + beta * norm(A) s`. Two link types:

- **citations**: one article refers to another in the law text;
- **sections**: articles in the same section of the legal tree.

The strengths `beta` were chosen on dev by nDCG@10 (`run_baselines.py --sweep`),
so the dev gains here are optimistic.

In [13]:
pd.concat([
    run('graph_cite', spread(scores['hybrid_rrf'], (cite, CONFIG.beta_cite))),
    run('graph_tree', spread(scores['hybrid_rrf'], (tree, CONFIG.beta_tree))),
    run('graph_both', spread(scores['hybrid_rrf'], (cite, CONFIG.beta_both[0]),
                             (tree, CONFIG.beta_both[1]))),
])

,mrr,recall@1,recall@5,recall@10,recall@20,ndcg@10
graph_cite,0.370,0.119,0.339,0.495,0.627,0.336
graph_tree,0.388,0.110,0.319,0.485,0.684,0.333
graph_both,0.401,0.124,0.341,0.493,0.694,0.347


## 7. Comparison table

Same layout as table 1 in the notes. Best value per column in bold.

In [14]:
LABELS = {
    'random': 'Random',
    'bm25': 'BM25 (keywords)',
    'dense': 'Dense (BGE-M3)',
    'hybrid': 'BM25 + dense, z-score fusion',
    'hybrid_rrf': 'BM25 + dense, rank fusion (RRF)',
    'graph_cite': 'RRF + spreading along citations',
    'graph_tree': 'RRF + spreading within legal sections',
    'graph_both': 'RRF + spreading along both',
}
METRICS = ['mrr', 'recall@1', 'recall@5', 'recall@10', 'recall@20', 'ndcg@10']
HEADERS = ['MRR', 'Recall@1', 'Recall@5', 'Recall@10', 'Recall@20', 'nDCG@10']

table = pd.DataFrame([summary[s] for s in LABELS], index=list(LABELS.values()))[METRICS]
table.columns = HEADERS
table.index.name = 'System'
table.insert(0, 'Uses atoms?', 'No')
table.style.format(precision=2, subset=HEADERS).highlight_max(subset=HEADERS, props='font-weight: bold')

,Uses atoms?,MRR,Recall@1,Recall@5,Recall@10,Recall@20,nDCG@10
System,,,,,,,
Random,No,0.10,0.01,0.06,0.11,0.23,0.06
BM25 (keywords),No,0.33,0.08,0.27,0.46,0.61,0.29
Dense (BGE-M3),No,0.31,0.08,0.31,0.42,0.61,0.28
"BM25 + dense, z-score fusion",No,0.37,0.10,0.29,0.46,0.66,0.31
"BM25 + dense, rank fusion (RRF)",No,0.38,0.10,0.30,0.46,0.66,0.32
RRF + spreading along citations,No,0.37,0.12,0.34,0.50,0.63,0.34
RRF + spreading within legal sections,No,0.39,0.11,0.32,0.48,0.68,0.33
RRF + spreading along both,No,0.40,0.12,0.34,0.49,0.69,0.35


In [15]:
def to_markdown(t):
    # Markdown table for the notes, two decimals, best value per column in bold
    nums = t[HEADERS].round(2)
    lines = ['| ' + ' | '.join([t.index.name, *t.columns]) + ' |',
             '|' + '---|' * (len(t.columns) + 1)]
    for name, row in t.iterrows():
        cells = [f'**{v:.2f}**' if v == nums[c].max() else f'{v:.2f}'
                 for c, v in nums.loc[name].items()]
        lines.append('| ' + ' | '.join([name, row['Uses atoms?'], *cells]) + ' |')
    return '\n'.join(lines)


print(to_markdown(table))

| System | Uses atoms? | MRR | Recall@1 | Recall@5 | Recall@10 | Recall@20 | nDCG@10 |
|---|---|---|---|---|---|---|---|
| Random | No | 0.10 | 0.01 | 0.06 | 0.11 | 0.23 | 0.06 |
| BM25 (keywords) | No | 0.33 | 0.08 | 0.27 | 0.46 | 0.61 | 0.29 |
| Dense (BGE-M3) | No | 0.31 | 0.08 | 0.31 | 0.42 | 0.61 | 0.28 |
| BM25 + dense, z-score fusion | No | 0.37 | 0.10 | 0.29 | 0.46 | 0.66 | 0.31 |
| BM25 + dense, rank fusion (RRF) | No | 0.38 | 0.10 | 0.30 | 0.46 | 0.66 | 0.32 |
| RRF + spreading along citations | No | 0.37 | **0.12** | **0.34** | **0.50** | 0.63 | 0.34 |
| RRF + spreading within legal sections | No | 0.39 | 0.11 | 0.32 | 0.48 | 0.68 | 0.33 |
| RRF + spreading along both | No | **0.40** | **0.12** | **0.34** | 0.49 | **0.69** | **0.35** |


In [16]:
out = PROJECT_ROOT / 'outputs' / 'tables' / f'baselines_{SPLIT}_comparison.csv'
table.round(4).to_csv(out)
print(f'-> {out.relative_to(PROJECT_ROOT)}')

-> outputs\tables\baselines_dev_comparison.csv


## 8. Are the differences real?

With 125 questions, small gaps can be noise. For each pair: the mean
per-question difference, a bootstrap 95% interval (10,000 resamples of
questions) and a paired sign-flip permutation p-value (10,000 draws), as in
the decisions log.

In [17]:
def paired_test(a, b, metric, draws=10_000, seed=0):
    # Mean of (a - b) over questions, bootstrap 95% interval, two-sided permutation p
    rng = np.random.default_rng(seed)
    d = (per_query[a][metric] - per_query[b][metric]).to_numpy()
    boot = d[rng.integers(0, len(d), (draws, len(d)))].mean(axis=1)
    flips = (rng.choice([-1, 1], (draws, len(d))) * d).mean(axis=1)
    p = (np.sum(np.abs(flips) >= abs(d.mean())) + 1) / (draws + 1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'diff': d.mean(), '95% low': lo, '95% high': hi, 'p': p}


PAIRS = [('dense', 'bm25'), ('hybrid', 'bm25'), ('hybrid_rrf', 'bm25'),
         ('hybrid_rrf', 'dense'), ('hybrid_rrf', 'hybrid'),
         ('graph_cite', 'hybrid_rrf'), ('graph_tree', 'hybrid_rrf'),
         ('graph_both', 'hybrid_rrf')]
tests = pd.DataFrame([{'comparison': f'{a} - {b}', 'metric': m, **paired_test(a, b, m)}
                      for a, b in PAIRS for m in ['mrr', 'recall@10', 'ndcg@10']])
tests.pivot(index='comparison', columns='metric').swaplevel(axis=1).sort_index(axis=1).round(3)

metric                       mrr                        ndcg@10                       recall@10                      
                        95% high 95% low   diff      p 95% high 95% low   diff      p  95% high 95% low   diff      p
comparison                                                                                                           
dense - bm25               0.054  -0.078 -0.012  0.721    0.045  -0.070 -0.012  0.666     0.039  -0.134 -0.048  0.288
graph_both - hybrid_rrf    0.057  -0.005  0.025  0.120    0.051   0.009  0.030  0.005     0.064   0.001  0.030  0.073
graph_cite - hybrid_rrf    0.041  -0.054 -0.006  0.802    0.050  -0.013  0.019  0.264     0.073  -0.003  0.032  0.103
graph_tree - hybrid_rrf    0.031  -0.006  0.012  0.226    0.028   0.004  0.015  0.018     0.047   0.001  0.022  0.063
hybrid - bm25              0.079   0.004  0.042  0.034    0.049  -0.016  0.017  0.319     0.057  -0.062 -0.003  0.933
hybrid_rrf - bm25          0.102  -0.001  0.051  0.061    0.064  -0.017  0.025  0.238     0.068  -0.069 -0.000  0.997
hybrid_rrf - dense         0.112   0.015  0.063  0.012    0.077  -0.005  0.037  0.085     0.116  -0.022  0.047  0.187
hybrid_rrf - hybrid        0.042  -0.024  0.009  0.589    0.027  -0.012  0.008  0.431     0.042  -0.037  0.002  0.914

## 9. Check against `run_baselines.py`

Same functions, settings and seeds, so the notebook should reproduce the
script's last saved dev run.

In [18]:
saved_path = PROJECT_ROOT / 'outputs' / 'tables' / f'baselines_{SPLIT}_summary.json'
if saved_path.exists():
    saved = json.loads(saved_path.read_text(encoding='utf-8'))['results']
    diff = (pd.DataFrame(summary).T - pd.DataFrame(saved).T).loc[list(saved)][METRICS]
    print('max abs difference vs saved run:', float(diff.abs().max().max()))
else:
    print('no saved run to compare against')

max abs difference vs saved run: 0.0
